This notebooks includes some experiments included in Section PRNU based source camera identification of the article. Namely it allows the user to perform source camera identification using the python implementation extracted from https://github.com/sim-pez/prnu/tree/main of the he Binghamton Toolbox originally implemented in Matlab. 

This code provides the following results:
1. PRNU and flat fields JPG, spce and pearson correlation violin diagrams.
2. PRNU and flat fields RAW, spce and pearson correlation violin diagrams.

## PRNU and flat fields JPG

In [ ]:
import os
from glob import glob
from multiprocessing import cpu_count, Pool

import numpy as np
from PIL import Image
import time

import sys
sys.path.append("./prnu")
import prnu
import matplotlib.pyplot as plt
import argparse

import rawpy

from sklearn.metrics import roc_curve, auc

from collections import Counter

In [ ]:
base_dir = PATH
tags = sorted(os.listdir(base_dir))  # ['A1', 'A2', ...]
jpeg_ext = 'jpg'

flat_images, flat_devices = [], []

for tag in tags:
    flat_dir = os.path.join(base_dir, tag, 'flat', 'jpg')
    
    # Flats
    for img_path in glob(os.path.join(flat_dir, f'*.{jpeg_ext}')):
        device = os.path.split(img_path)[1].split('_')[0]
        flat_images.append(img_path)
        flat_devices.append(device)
        

flat_images = np.array(flat_images)
flat_devices = np.array(flat_devices)

In [ ]:
print('Computing fingerprints (using 50 flats per device)')

fingerprint_devices = sorted(np.unique(flat_devices))

k = []
eval_images = []
eval_devices = []

for device in fingerprint_devices:
    print(device)
    
    device_imgs = flat_images[flat_devices == device]
    
    
    # Split
    train_imgs = device_imgs[:50]
    test_imgs = device_imgs[50:100]
    
    imgs = []
    for img_path in train_imgs:
        im = np.asarray(Image.open(img_path))
        if im.dtype != np.uint8 or im.ndim != 3:
            print('Skipping image:', img_path)
            continue
        imgs.append(prnu.cut_ctr(im, (512, 512, 3)))
    
    # Fingerprint
    k.append(prnu.extract_multiple_aligned(imgs, processes=1))
    
    # Eval
    for img_path in test_imgs:
        eval_images.append(img_path)
        eval_devices.append(device)

k = np.stack(k, 0)

In [ ]:
print('Computing residuals (evaluation on flats)')

w = []
for img_path in eval_images:
    im = np.asarray(Image.open(img_path))
    w.append(prnu.cut_ctr(im, (512, 512, 3)))

w = np.stack([prnu.extract_single(img) for img in w], 0)

eval_devices = np.array(eval_devices)

In [ ]:
matching_cases = []
mismatching_cases = []

for f_idx, fingerprint_k in enumerate(k):
    device_fp = fingerprint_devices[f_idx]

    for n_idx, natural_w in enumerate(w):
        device_nat = eval_devices[n_idx]

        cc2d = prnu.crosscorr_2d(fingerprint_k, natural_w)
        pce_value = prnu.pce(cc2d)['pce']

        case = {
            'pce': pce_value,
            'fingerprint_device': device_fp,
            'test_device': device_nat,
            'test_path': eval_images[n_idx]
        }

        if device_fp == device_nat:
            matching_cases.append(case)
        else:
            mismatching_cases.append(case)

In [ ]:
np.savez(
    "pce_prnu-flatfields-jpg.npz",
    matching_cases=matching_cases,
    mismatching_cases=mismatching_cases
)

In [ ]:
data = np.load("pce_prnu-flatfields-jpg.npzz", allow_pickle=True)
matching_cases = data["matching_cases"].tolist()
mismatching_cases = data["mismatching_cases"].tolist()

from collections import defaultdict

data = defaultdict(lambda: {'match': [], 'mismatch': []})

# Matching
for case in matching_cases:
    sensor = case['fingerprint_device']
    data[sensor]['match'].append(case['pce'])

# Mismatching
for case in mismatching_cases:
    sensor = case['fingerprint_device']
    data[sensor]['mismatch'].append(case['pce'])

sensors = sorted(data.keys())

match_values = [data[s]['match'] for s in sensors]
mismatch_values = [data[s]['mismatch'] for s in sensors]


plt.figure(figsize=(14,6))

positions_match = []
positions_mismatch = []

for i in range(len(sensors)):
    positions_match.append(i - 0.15)
    positions_mismatch.append(i + 0.15)

# Matching
vp1 = plt.violinplot(match_values, positions=positions_match, widths=0.25, showmeans=True)
for body in vp1['bodies']:
    body.set_facecolor('green')
    body.set_alpha(0.6)

# Mismatching
vp2 = plt.violinplot(mismatch_values, positions=positions_mismatch, widths=0.25, showmeans=True)
for body in vp2['bodies']:
    body.set_facecolor('red')
    body.set_alpha(0.6)


plt.xticks(range(len(sensors)), sensors, rotation=45)
plt.yscale('log')
plt.xlabel("Sensor", fontsize =16)
plt.ylabel("sPCE value", fontsize = 16)
plt.title("sPCE distribution per sensor PRNU using flat fields", fontsize = 16)

plt.grid(True, which="both", linestyle="--", linewidth=0.5)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

plt.legend([plt.Line2D([0],[0], color='green', lw=4),
            plt.Line2D([0],[0], color='red', lw=4)],
           ['Matching', 'Mismatching'], fontsize = 11)

plt.tight_layout()
plt.show()

In [ ]:
import os
from glob import glob
from multiprocessing import cpu_count, Pool

import numpy as np
from PIL import Image
import time

import sys
sys.path.append("./prnu")
import prnu
import matplotlib.pyplot as plt
import argparse
from scipy.stats import pearsonr

import rawpy

from sklearn.metrics import roc_curve, auc

from collections import Counter

In [ ]:
base_dir = PATH
tags = sorted(os.listdir(base_dir))  # ['A1', 'A2', ...]
jpeg_ext = 'jpg'

flat_images, flat_devices = [], []

for tag in tags:
    flat_dir = os.path.join(base_dir, tag, 'flat', 'jpg')
    
    # Flats
    for img_path in glob(os.path.join(flat_dir, f'*.{jpeg_ext}')):
        device = os.path.split(img_path)[1].split('_')[0]
        flat_images.append(img_path)
        flat_devices.append(device)
        

flat_images = np.array(flat_images)
flat_devices = np.array(flat_devices)

In [ ]:
print('Computing fingerprints (using 50 flats per device)')

fingerprint_devices = sorted(np.unique(flat_devices))

k = []
eval_images = []
eval_devices = []

for device in fingerprint_devices:
    print(device)
    
    device_imgs = flat_images[flat_devices == device]
    
    
    # Split
    train_imgs = device_imgs[:50]
    test_imgs = device_imgs[50:100]
    
    imgs = []
    for img_path in train_imgs:
        im = np.asarray(Image.open(img_path))
        if im.dtype != np.uint8 or im.ndim != 3:
            print('Skipping image:', img_path)
            continue
        imgs.append(prnu.cut_ctr(im, (512, 512, 3)))
    
    # Fingerprint
    k.append(prnu.extract_multiple_aligned(imgs, processes=1))
    
    # Eval
    for img_path in test_imgs:
        eval_images.append(img_path)
        eval_devices.append(device)

k = np.stack(k, 0)

In [ ]:
print('Computing residuals (evaluation on flats)')

w = []
for img_path in eval_images:
    im = np.asarray(Image.open(img_path))
    w.append(prnu.cut_ctr(im, (512, 512, 3)))

w = np.stack([prnu.extract_single(img) for img in w], 0)

eval_devices = np.array(eval_devices)

In [ ]:
def pearson_corr(a, b):
    a_flat = a.flatten()
    b_flat = b.flatten()
    return pearsonr(a_flat,b_flat)[0]

In [ ]:
matching_cases = []
mismatching_cases = []

for f_idx, fingerprint_k in enumerate(k):
    device_fp = fingerprint_devices[f_idx]

    for n_idx, natural_w in enumerate(w):
        device_nat = eval_devices[n_idx]

        pearson = pearson_corr(fingerprint_k,natural_w)

        case = {
            'pearson': pearson,
            'fingerprint_device': device_fp,
            'natural_device': device_nat,
            'natural_path': eval_devices[n_idx]
        }

        if device_fp == device_nat:
            matching_cases.append(case)
        else:
            mismatching_cases.append(case)

In [ ]:
np.savez(
    "pearson_prnu-flatfields-jpg.npz",
    matching_cases=matching_cases,
    mismatching_cases=mismatching_cases
)

In [ ]:
data = np.load("pearson_prnu-flatfields-jpg.npz", allow_pickle=True)
matching_cases = data["matching_cases"].tolist()
mismatching_cases = data["mismatching_cases"].tolist()

from collections import defaultdict

data = defaultdict(lambda: {'match': [], 'mismatch': []})

# Matching
for case in matching_cases:
    sensor = case['fingerprint_device']
    data[sensor]['match'].append(case['pearson'])

# Mismatching
for case in mismatching_cases:
    sensor = case['fingerprint_device']
    data[sensor]['mismatch'].append(case['pearson'])

sensors = sorted(data.keys())

match_values = [data[s]['match'] for s in sensors]
mismatch_values = [data[s]['mismatch'] for s in sensors]

plt.figure(figsize=(14,6))

positions_match = []
positions_mismatch = []

for i in range(len(sensors)):
    positions_match.append(i - 0.15)
    positions_mismatch.append(i + 0.15)

# Matching
vp1 = plt.violinplot(match_values, positions=positions_match, widths=0.25, showmeans=True)
for body in vp1['bodies']:
    body.set_facecolor('green')
    body.set_alpha(0.6)

# Mismatching
vp2 = plt.violinplot(mismatch_values, positions=positions_mismatch, widths=0.25, showmeans=True)
for body in vp2['bodies']:
    body.set_facecolor('red')
    body.set_alpha(0.6)

plt.xticks(range(len(sensors)), sensors, rotation=45)

plt.xlabel("Sensor", fontsize =16)
plt.ylabel("Pearson correlation value", fontsize = 16)
plt.title("Pearson correlation distribution per sensor PRNU in JPG for flatfields", fontsize = 16)

plt.grid(True, which="both", linestyle="--", linewidth=0.5)

plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

plt.legend([plt.Line2D([0],[0], color='green', lw=4),
            plt.Line2D([0],[0], color='red', lw=4)],
           ['Matching', 'Mismatching'], fontsize = 11)

plt.tight_layout()
plt.show()

## PRNU and flat fields raw

In [ ]:
import os
from glob import glob
from multiprocessing import cpu_count, Pool

import numpy as np
from PIL import Image
import time

import sys
sys.path.append("./prnu")
import prnu_raw
import matplotlib.pyplot as plt
import argparse
from scipy.stats import pearsonr

import rawpy

from sklearn.metrics import roc_curve, auc

from collections import Counter

In [ ]:
def split_bayer(im):
    R  = im[0::2, 0::2]
    G1 = im[0::2, 1::2]
    G2 = im[1::2, 0::2]
    B  = im[1::2, 1::2]
    return [R, G1, G2, B]

In [ ]:
def open_image(img_path):
    img = rawpy.imread(img_path)
    return img.raw_image_visible

In [ ]:
def pearson_corr(a, b):
    a_flat = a.flatten()
    b_flat = b.flatten()
    return pearsonr(a_flat,b_flat)[0]

In [ ]:
base_dir = PATH
tags = sorted(os.listdir(base_dir))

raw_exts = ['cr3', 'cr2', 'arw']

flat_images, flat_devices = [], []

for tag in tags:
    flat_dir = os.path.join(base_dir, tag, 'flat', 'raw')
    
    for ext in raw_exts:
        for img_path in glob(os.path.join(flat_dir, f'*.{ext}')):
            device = os.path.split(img_path)[1].split('_')[0]
            flat_images.append(img_path)
            flat_devices.append(device)

flat_images = np.array(flat_images)
flat_devices = np.array(flat_devices)

In [ ]:
print('Computing fingerprints (50 flats per device)')

fingerprint_devices = sorted(np.unique(flat_devices))

k = []  # [num_devices, 4, H, W]

eval_images = []
eval_devices = []

for device in fingerprint_devices:
    print(device)
    
    device_imgs = flat_images[flat_devices == device]
    
    # Split 50 / rest
    train_imgs = device_imgs[:50]
    test_imgs = device_imgs[50:]
    
    channel_imgs = [[], [], [], []]  # R, G1, G2, B
    
    for img_path in train_imgs:
        im = np.asarray(open_image(img_path))
        channels = split_bayer(im)
        
        for c in range(4):
            cut = prnu_raw.cut_ctr(channels[c], (512, 512))
            channel_imgs[c].append(cut)
    
    k_device = []
    for c in range(4):
        k_c = prnu_raw.extract_multiple_aligned(channel_imgs[c], processes=1)
        k_device.append(k_c)
    
    k.append(k_device)
    

    for img_path in test_imgs:
        eval_images.append(img_path)
        eval_devices.append(device)

k = np.array(k)  # [num_devices, 4, H, W]
eval_images = np.array(eval_images)
eval_devices = np.array(eval_devices)

In [ ]:
print('Computing residuals (evaluation flats)')

w = []  # [num_images, 4, H, W]

for img_path in eval_images:
    im = np.asarray(open_image(img_path))
    channels = split_bayer(im)
    
    w_channels = []
    for c in range(4):
        cut = prnu_raw.cut_ctr(channels[c], (512, 512))
        w_c = prnu_raw.extract_single(cut)
        w_channels.append(w_c)
    
    w.append(w_channels)

w = np.array(w)

In [ ]:
print('Evaluating (PCE + Pearson)')

results = []

channel_names = ['R', 'G1', 'G2', 'B']

for f_idx, fingerprint_k in enumerate(k):
    device_fp = fingerprint_devices[f_idx]
    
    for n_idx, natural_w in enumerate(w):
        device_nat = eval_devices[n_idx]
        
        for c in range(4):
            cc2d = prnu_raw.crosscorr_2d(fingerprint_k[c], natural_w[c])
            pce_value = prnu_raw.pce(cc2d)['pce']
            
            pearson = pearson_corr(fingerprint_k[c], natural_w[c])
            
            results.append({
                'pce': pce_value,
                'pearson': pearson,
                'match': int(device_fp == device_nat),
                'fp_device': device_fp,
                'eval_device': device_nat,
                'img_idx': n_idx,
                'channel': channel_names[c]
            })

In [ ]:
np.savez('spce-pearson-raw-flatfields.npz', results=results)

In [ ]:
output_dir = "raw-violins-prnu-ff"
os.makedirs(output_dir, exist_ok=True)

data = np.load('spce-pearson-raw-flatfields.npzz', allow_pickle=True)
results = data['results']

channel_names = sorted(list(set(case['channel'] for case in results)))

for cname in channel_names:

    sensor_data_pce = defaultdict(lambda: {'match': [], 'mismatch': []})
    sensor_data_pearson = defaultdict(lambda: {'match': [], 'mismatch': []})

    for case in results:
        if case['channel'] != cname:
            continue

        sensor = case['fp_device']
        
        if case['match']:
            sensor_data_pce[sensor]['match'].append(case['pce'])
            sensor_data_pearson[sensor]['match'].append(case['pearson'])
        else:
            sensor_data_pce[sensor]['mismatch'].append(case['pce'])
            sensor_data_pearson[sensor]['mismatch'].append(case['pearson'])

    sensors = sorted(sensor_data_pce.keys())

    positions_match = [i - 0.15 for i in range(len(sensors))]
    positions_mismatch = [i + 0.15 for i in range(len(sensors))]

    match_values = [sensor_data_pce[s]['match'] for s in sensors]
    mismatch_values = [sensor_data_pce[s]['mismatch'] for s in sensors]

    plt.figure(figsize=(14,6))

    vp1 = plt.violinplot(match_values, positions=positions_match, widths=0.25, showmeans=True)
    for body in vp1['bodies']:
        body.set_facecolor('green')
        body.set_alpha(0.6)

    vp2 = plt.violinplot(mismatch_values, positions=positions_mismatch, widths=0.25, showmeans=True)
    for body in vp2['bodies']:
        body.set_facecolor('red')
        body.set_alpha(0.6)

    plt.xticks(range(len(sensors)), sensors, rotation=45)
    plt.yscale('log')
    plt.xlabel("Sensor", fontsize=16)
    plt.ylabel("sPCE value", fontsize=16)
    plt.title(f"sPCE distribution per sensor - Channel {cname} PRNU flat fields", fontsize=16)

    plt.grid(True, which="both", linestyle="--", linewidth=0.5)

    plt.legend([plt.Line2D([0],[0], color='green', lw=4),
                plt.Line2D([0],[0], color='red', lw=4)],
               ['Matching', 'Mismatching'], fontsize=11)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"pce_{cname}.png"), dpi=300)
    plt.close()

    match_values = [sensor_data_pearson[s]['match'] for s in sensors]
    mismatch_values = [sensor_data_pearson[s]['mismatch'] for s in sensors]

    plt.figure(figsize=(14,6))

    vp1 = plt.violinplot(match_values, positions=positions_match, widths=0.25, showmeans=True)
    for body in vp1['bodies']:
        body.set_facecolor('green')
        body.set_alpha(0.6)

    vp2 = plt.violinplot(mismatch_values, positions=positions_mismatch, widths=0.25, showmeans=True)
    for body in vp2['bodies']:
        body.set_facecolor('red')
        body.set_alpha(0.6)

    plt.xticks(range(len(sensors)), sensors, rotation=45)
    plt.xlabel("Sensor", fontsize=16)
    plt.ylabel("Pearson correlation", fontsize=16)
    plt.title(f"Pearson distribution per sensor - Channel {cname} PRNU flat fields", fontsize=16)

    plt.grid(True, linestyle="--", linewidth=0.5)

    plt.legend([plt.Line2D([0],[0], color='green', lw=4),
                plt.Line2D([0],[0], color='red', lw=4)],
               ['Matching', 'Mismatching'], fontsize=11)

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"pearson_{cname}.png"), dpi=300)
    plt.close()